# Phase 2: Fine-Tuning Cross-Encoder Reranker (`BAAI/bge-reranker-base`)

This notebook fine-tunes the cross-encoder reranker with positive query-chunk pairs and mined hard negatives.

### Features:
- Checkpoints saved every 100 steps
- Automatic resumption if Kaggle session disconnects
- Outputs deployable reranker weights for Antigravity Enterprise RAG


In [ ]:
# 1. Install dependencies
!pip install -q sentence-transformers datasets accelerate
!nvidia-smi


In [ ]:
# 2. Setup and Paths
import os
import json
import glob
import random
import torch
from torch.utils.data import DataLoader
from sentence_transformers import InputExample
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

DATASET_DIR = '/kaggle/input/enterprise-rag-dataset'
CHECKPOINT_DIR = '/kaggle/working/checkpoints_reranker'
FINAL_OUTPUT_DIR = '/kaggle/working/reranker_adapter'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)


In [ ]:
# 3. Load Corpus and Construct Positive + Hard Negative Pairs
doc_text_map = {}
chunks_path = os.path.join(DATASET_DIR, 'chunks.jsonl')
if not os.path.exists(chunks_path):
    found = glob.glob(f'{DATASET_DIR}/**/chunks.jsonl', recursive=True)
    if found: chunks_path = found[0]

with open(chunks_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        c = json.loads(line)
        cid = c.get('chunk_id')
        did = c.get('document_id')
        txt = c.get('text', '')
        if cid: doc_text_map[cid] = txt
        if did and did not in doc_text_map: doc_text_map[did] = txt

all_doc_ids = list(doc_text_map.keys())
print(f'Total available corpus documents: {len(doc_text_map):,}')

questions_path = os.path.join(DATASET_DIR, 'questions.jsonl')
if not os.path.exists(questions_path):
    found = glob.glob(f'{DATASET_DIR}/**/questions.jsonl', recursive=True)
    if found: questions_path = found[0]

with open(questions_path, 'r', encoding='utf-8') as f:
    lines = [json.loads(l.strip()) for l in f if l.strip()]

train_samples = []
eval_samples = []
split_idx = int(len(lines) * 0.85)

for i, item in enumerate(lines):
    q = item.get('question')
    pos_ids = set(item.get('expected_doc_ids', []))
    target_list = train_samples if i < split_idx else eval_samples

    # Add Positive Pair (label 1.0)
    for pid in pos_ids:
        if pid in doc_text_map:
            target_list.append(InputExample(texts=[q, doc_text_map[pid]], label=1.0))
            
            # Add Random/Hard Negative Pair from corpus (label 0.0)
            neg_id = random.choice(all_doc_ids)
            while neg_id in pos_ids:
                neg_id = random.choice(all_doc_ids)
            target_list.append(InputExample(texts=[q, doc_text_map[neg_id]], label=0.0))

print(f'Constructed {len(train_samples):,} training pairs (50% positive, 50% negative).')
print(f'Constructed {len(eval_samples):,} evaluation pairs.')


In [ ]:
# 4. Initialize Model and Check for Checkpoint
BASE_MODEL_NAME = 'BAAI/bge-reranker-base'

# Look for latest checkpoint or completed model to resume/continue
checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'), key=os.path.getmtime)
if checkpoints:
    latest_checkpoint = checkpoints[-1]
    print(f'🔄 Resuming CrossEncoder from: {latest_checkpoint}')
    model = CrossEncoder(latest_checkpoint, num_labels=1, max_length=512, device=DEVICE)
elif os.path.exists(FINAL_OUTPUT_DIR) and os.listdir(FINAL_OUTPUT_DIR):
    print(f'🔄 Resuming from completed reranker: {FINAL_OUTPUT_DIR}')
    model = CrossEncoder(FINAL_OUTPUT_DIR, num_labels=1, max_length=512, device=DEVICE)
else:
    print(f'Initializing CrossEncoder from: {BASE_MODEL_NAME}')
    model = CrossEncoder(BASE_MODEL_NAME, num_labels=1, max_length=512, device=DEVICE)


In [ ]:
# 5. Execute Reranker Training with Checkpoints
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=8)
evaluator = CEBinaryClassificationEvaluator.from_input_examples(eval_samples, name='eval_reranker')

EPOCHS = 3
WARMUP_STEPS = int(len(train_dataloader) * EPOCHS * 0.1)

model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path=CHECKPOINT_DIR,
    evaluation_steps=100,
    optimizer_params={'lr': 1e-5},
    show_progress_bar=True
)

model.save(FINAL_OUTPUT_DIR)
print(f'✅ Reranker fine-tuning complete! Saved to: {FINAL_OUTPUT_DIR}')
!ls -lh /kaggle/working/reranker_adapter
